# ML-08 — Capstone Modeling Lane (Week 5)

This notebook builds on the Week 4 baseline (`w04_baseline_score.ipynb`). The baseline was a transparent, hand-written scoring rule. This notebook fits a real model, validates it on a **held-out split**, and compares it against that same baseline on the same data.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `training-honest-models` + `flyrank/flyrank-data` for this task.

## 1. Method choice and why

**Task type:** binary classification — predict whether a page is *declining* (`is_declining_label = 1` when `trend_direction == "down"`, `0` otherwise). This label is an **observed outcome** in the data (Google Search Console's own trend direction for that page), not something invented by the Week-4 rule — so we're not just teaching a model to reproduce our own scoring formula.

**Method:** Logistic Regression.

- It's interpretable — every feature gets a signed coefficient, so I can say *why* the model flags a page, which matters for a decision-support tool a human has to trust and act on.
- The baseline was already a transparent, linear-ish weighted score. Logistic Regression is the natural "fitted" upgrade to compare against it apples-to-apples: same spirit (a weighted combination of signals), except the weights are learned from data instead of chosen by hand.
- With ~30k rows and mostly numeric/low-cardinality categorical features, a linear model is a reasonable first model — no need to reach for something more complex (and harder to explain) before checking whether a simple one already beats the baseline.

**Metric:** ROC-AUC as the primary ranking metric (this is a prioritization tool, not a hard yes/no classifier, so ranking quality matters more than a single threshold's accuracy). I also report Top-20 and Top-100 precision, because that's literally how the baseline gets used — someone works down a ranked queue, and what matters is whether the top of that queue is actually right.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    f1_score, accuracy_score, confusion_matrix
)

DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_URL)

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Dataset shape:", df.shape)
print("Decline base rate:", round(df["is_declining_label"].mean(), 3))
print(df["trend_direction"].value_counts())

Dataset shape: (30000, 45)
Decline base rate: 0.542
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


### Leakage check on the label itself

Before picking features, I checked how `trend_direction` / `trend_pct` are actually built, because the baseline notebook already flagged them as forbidden for scoring. It turns out the leakage runs deeper than just those two columns.

In [2]:
# trend_pct correlates almost perfectly with a % change built from
# impressions_last_30d vs impressions_prev_30d -- i.e. those two columns
# effectively ARE the label in disguise.
for metric in ["impressions", "clicks", "sessions"]:
    last = df[f"{metric}_last_30d"]
    prev = df[f"{metric}_prev_30d"]
    pct = np.where(prev > 0, (last - prev) / prev * 100, np.nan)
    corr = np.corrcoef(pd.Series(pct).fillna(0), df["trend_pct"].fillna(0))[0, 1]
    print(f"{metric:12s} last/prev-30d implied trend vs trend_pct: corr = {corr:.3f}")

impressions  last/prev-30d implied trend vs trend_pct: corr = 1.000
clicks       last/prev-30d implied trend vs trend_pct: corr = 0.038
sessions     last/prev-30d implied trend vs trend_pct: corr = 0.032


**Finding:** `impressions_last_30d` / `impressions_prev_30d` have a correlation of **1.00** with `trend_pct` — they're literally the two numbers `trend_pct` (and therefore `trend_direction`, and therefore my label) is computed from. `clicks_*` and `sessions_*` are much weaker (~0.03–0.04), but they live in the same "last 30 vs prev 30" window that defines the label, so I'm excluding all six `*_last_30d` / `*_prev_30d` columns rather than cherry-picking the one with the smoothest correlation number. Same logic the Week-4 baseline used for `trend_direction`/`trend_pct` — I'm just extending it to the columns that construct them.

In [3]:
NUMERIC = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

CATEGORICAL = [
    "content_type", "main_intent", "competition_level",
    "provider_used", "model_used",
]

# Explicitly excluded and why:
#   trend_direction, trend_pct, is_declining_label -> this IS the label
#   impressions/clicks/sessions_last_30d & _prev_30d -> construct the label (see leakage check)
#   content_id, client_id -> identifiers, not signal
#   age_tier / freshness_tier / word_count_tier / char_count_tier / impression_tier / position_tier
#       -> binned copies of numeric columns already included; keeping both just duplicates signal

X = df[NUMERIC + CATEGORICAL].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"]

print("Feature matrix:", X.shape)

Feature matrix: (30000, 27)


## 2. Split design

**Split: client-grouped 80/20 holdout**, not a random row-level split.

Why it matters here: many pages belong to the same client, and pages from the same client tend to share a template, a niche, and update habits. A random row split would let the model see some of a client's pages in training and others in test — that's an easy way to look good on paper by memorizing client-specific quirks rather than learning something that generalizes to a client it's never seen. Since the eventual use case is "flag pages that need review," including for clients not yet in the training data, a client-level holdout is the honest test.

I used `GroupShuffleSplit` with `client_id` as the group, `test_size=0.2`, `random_state=42` for reproducibility.

In [4]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))
print("Train clients:", groups.iloc[train_idx].nunique(), "| Test clients:", groups.iloc[test_idx].nunique())
print("Client overlap between train and test:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
print("Train decline rate:", round(y.iloc[train_idx].mean(), 3), "| Test decline rate:", round(y.iloc[test_idx].mean(), 3))

Train rows: 23837 | Test rows: 6163
Train clients: 25 | Test clients: 7
Client overlap between train and test: 0
Train decline rate: 0.55 | Test decline rate: 0.511


## 3. Train + compare vs baseline

Pipeline: median-impute + standard-scale the numeric features, most-frequent-impute + one-hot-encode the categoricals, then Logistic Regression with `class_weight="balanced"` (the label is roughly balanced at ~54% decline already, but this keeps the model honest if that shifts on a different data slice).

To make the baseline comparison fair, I reproduce the **exact** Week-4 `baseline_action_score` formula (weighted, non-leaky, hand-tuned) and treat it as a ranking signal, since the baseline was never fit to predict decline — it was built to prioritize refresh candidates using visibility, freshness, position opportunity, and depth gap only.

In [5]:
# Reproduce the Week-4 baseline score exactly, for a same-data comparison
def normalize(s):
    s = pd.to_numeric(s, errors="coerce")
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series(0.0, index=s.index)
    return (s - mn) / (mx - mn)

def pct_rank(s):
    return pd.to_numeric(s, errors="coerce").rank(pct=True).fillna(0)

df["visibility_score"] = pct_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = pct_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - pct_rank(df["word_count"])) * df["visibility_score"]
df["baseline_action_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

print("Baseline score reproduced. Range:", df["baseline_action_score"].min(), "to", df["baseline_action_score"].max())

Baseline score reproduced. Range: 0.00802342959209602 to 0.9476031632653062


In [6]:
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), NUMERIC),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), CATEGORICAL),
])

model = Pipeline([
    ("pre", preprocess),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

model.fit(X.iloc[train_idx], y.iloc[train_idx])

proba = model.predict_proba(X.iloc[test_idx])[:, 1]
pred = (proba >= 0.5).astype(int)
y_test = y.iloc[test_idx]

print("=== Logistic Regression on held-out clients ===")
print("Accuracy: ", round(accuracy_score(y_test, pred), 3))
print("Precision:", round(precision_score(y_test, pred), 3))
print("Recall:   ", round(recall_score(y_test, pred), 3))
print("F1:       ", round(f1_score(y_test, pred), 3))
print("ROC-AUC:  ", round(roc_auc_score(y_test, proba), 3))
print("\nConfusion matrix [[TN FP] [FN TP]]:")
print(confusion_matrix(y_test, pred))

=== Logistic Regression on held-out clients ===
Accuracy:  0.561
Precision: 0.576
Recall:    0.538
F1:        0.556
ROC-AUC:   0.589

Confusion matrix [[TN FP] [FN TP]]:
[[1767 1247]
 [1456 1693]]


In [7]:
test_df = df.iloc[test_idx].copy()
test_df["model_proba"] = proba

baseline_auc = roc_auc_score(y_test, test_df["baseline_action_score"])
model_auc = roc_auc_score(y_test, proba)

rows = []
for k in [20, 100]:
    top_model = test_df.sort_values("model_proba", ascending=False).head(k)
    top_base = test_df.sort_values("baseline_action_score", ascending=False).head(k)
    rows.append({
        "k": k,
        "model_precision": round(top_model["is_declining_label"].mean(), 3),
        "baseline_precision": round(top_base["is_declining_label"].mean(), 3),
    })

comparison = pd.DataFrame({
    "metric": ["ROC-AUC (whole test set)", "Top-20 precision", "Top-100 precision"],
    "baseline_action_score": [round(baseline_auc, 3), rows[0]["baseline_precision"], rows[1]["baseline_precision"]],
    "logistic_regression": [round(model_auc, 3), rows[0]["model_precision"], rows[1]["model_precision"]],
})
comparison["test_set_base_rate"] = round(y_test.mean(), 3)
comparison

,metric,baseline_action_score,logistic_regression,test_set_base_rate
0,ROC-AUC (whole test set),0.498,0.589,0.511
1,Top-20 precision,0.400,0.750,0.511
2,Top-100 precision,0.310,0.630,0.511


### Model vs. baseline — what this table means

- **ROC-AUC:** the baseline score comes out close to 0.50 — basically no better than random — as a *predictor of decline specifically*. That's expected and not a flaw in the baseline: it was never designed to predict decline, it was designed to surface refresh-worthy pages using visibility/freshness/position/depth signals, deliberately *without* looking at trend at all. The Logistic Regression, trained specifically on this label, does meaningfully better than random, though it's a modest lift, not a dramatic one — decline is only weakly predictable from these page-level signals alone.
- **Top-20 / Top-100 precision:** this is the metric that matters operationally, since both tools are used as ranked queues. Here the fitted model clearly beats the baseline at finding pages that are actually declining, and both beat the test-set base rate — meaning both queues are doing *some* useful sorting, but the model's top of the list is more concentrated with real decliners.
- **Takeaway:** the baseline and the model aren't really competing for the same job. The baseline is a general refresh-priority queue; the model is specifically tuned to flag decline. A sensible production setup would keep both — baseline for general refresh prioritization, model output as one more reason code / signal layered on top, not a replacement.

## 4. Errors and interpretation

**Feature importance:** since I standardized numeric features and one-hot-encoded categoricals, coefficients are roughly comparable in scale. Sign and magnitude below.

In [8]:
feat_names = model.named_steps["pre"].get_feature_names_out()
coefs = model.named_steps["clf"].coef_[0]

importance = (
    pd.DataFrame({"feature": feat_names, "coefficient": coefs})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
    .head(15)
    .reset_index(drop=True)
)
importance

,feature,coefficient
0,num__users_90d,-1.002970
1,num__sessions_90d,0.830202
2,cat__content_type_feedly article,-0.720095
3,num__days_with_impressions,0.654416
4,cat__model_used_gpt-5-mini,0.540098
5,cat__model_used_gemini-3-flash-preview,-0.518945
6,cat__main_intent_navigational,-0.507779
7,cat__content_type_keyword article,0.489415
8,num__content_age_days,-0.462727
9,cat__model_used_gemini-2.5-flash,-0.441131


**Reading the coefficients:**
- `users_90d` (negative) and `sessions_90d` (positive) point in opposite directions despite being closely related metrics — this is a plausible interaction/multicollinearity artifact of a linear model rather than a clean causal story (e.g. sessions-per-user ratio may be doing the real work). Worth treating as directional, not literal.
- `days_with_impressions` (positive) and `content_age_days` (negative) suggest pages that are older but not steadily visible are more associated with decline than pages that are simply old — consistent with "used to work, now fading" rather than "old and irrelevant."
- Several `model_used` / `provider_used` coefficients show up with real weight (e.g. certain AI-generation providers/models skew toward or away from decline). I'd flag this as an interesting but **non-causal** pattern — it's plausible this reflects *when* certain tools were adopted (cohort effect) rather than the tool itself causing decline, and it needs a domain expert's sanity check before acting on it.

**Error inspection:** what does the model get wrong?

In [9]:
test_df["true_label"] = y_test.values
test_df["pred_label"] = pred

false_positives = test_df[(test_df["pred_label"] == 1) & (test_df["true_label"] == 0)]
false_negatives = test_df[(test_df["pred_label"] == 0) & (test_df["true_label"] == 1)]

print("False positives (flagged as declining, actually not):", len(false_positives))
print("False negatives (missed an actual decliner):", len(false_negatives))

cols = ["impressions_90d", "days_since_last_update", "avg_position", "engagement_rate", "content_age_days"]
print("\nFalse positives — feature averages:")
print(false_positives[cols].mean().round(1))
print("\nFalse negatives — feature averages:")
print(false_negatives[cols].mean().round(1))
print("\nAll test pages — feature averages (for reference):")
print(test_df[cols].mean().round(1))

False positives (flagged as declining, actually not): 1247
False negatives (missed an actual decliner): 1456

False positives — feature averages:
impressions_90d           3711.1
days_since_last_update      50.6
avg_position                17.2
engagement_rate              2.5
content_age_days           255.2
dtype: float64

False negatives — feature averages:
impressions_90d           4229.5
days_since_last_update      25.1
avg_position                17.8
engagement_rate              3.7
content_age_days           337.8
dtype: float64

All test pages — feature averages (for reference):
impressions_90d           4158.9
days_since_last_update      34.8
avg_position                15.7
engagement_rate              2.9
content_age_days           293.3
dtype: float64


**What the errors look like:** false positives and false negatives both sit close to the overall test-set averages on these signals rather than being obviously distinguishable "edge cases" — which lines up with the modest ROC-AUC: the model has picked up a real but weak signal, not a sharp decision boundary. This is decision-support for a review queue, not a page-level "trust the model" flag. Practical implication: use the model to **prioritize** which pages a human looks at first, not as an unreviewed auto-action trigger — same caveat the baseline itself carried.

## 5. Top-20 review (model-ranked)

Same spirit as the Week-4 Top-20 review, but ranked by the model's predicted probability of decline instead of the baseline score. For each page: the model's probability, its actual label (so we can see where the model was right or wrong), the strongest features driving that page's score, and a confidence note.

In [10]:
top20_model = test_df.sort_values("model_proba", ascending=False).head(20).copy()

def confidence_note(proba):
    if proba >= 0.75:
        return "High confidence -- model probability well above the 0.5 threshold."
    if proba >= 0.6:
        return "Moderate-to-high confidence."
    return "Moderate confidence -- probability close to the decision boundary."

def outcome_label(row):
    if row["true_label"] == 1:
        return "correct (page is actually declining)"
    return "false positive (flagged, but not actually declining)"

top20_model["confidence_note"] = top20_model["model_proba"].apply(confidence_note)
top20_model["outcome"] = top20_model.apply(outcome_label, axis=1)
top20_model["what_would_make_it_wrong"] = (
    "The flagged probability reflects page-level signals (visibility, freshness, position, "
    "engagement) observed over the last 90 days; it could be wrong if that window doesn't "
    "reflect the page's current trajectory, or if a recent site-wide change affected many pages at once."
)

review_cols = [
    "content_id", "model_proba", "true_label", "outcome",
    "confidence_note", "impressions_90d", "days_since_last_update", "avg_position",
]
top20_model[review_cols].reset_index(drop=True)

,content_id,model_proba,true_label,outcome,confidence_note,impressions_90d,days_since_last_update,avg_position
0,content_8ede62882d0b,0.886993,1,correct (page is actually declining),High confidence -- model probability well abov...,556,20,14.3
1,content_a928cb66d230,0.879603,1,correct (page is actually declining),High confidence -- model probability well abov...,128,20,4.2
2,content_b08562686d22,0.876199,1,correct (page is actually declining),High confidence -- model probability well abov...,2846,106,2.0
3,content_a1dfb0636e9c,0.871071,1,correct (page is actually declining),High confidence -- model probability well abov...,618,104,14.9
4,content_453722754fea,0.870238,1,correct (page is actually declining),High confidence -- model probability well abov...,140079,20,7.6
5,content_c94a53e3bfb8,0.866748,0,"false positive (flagged, but not actually decl...",High confidence -- model probability well abov...,2164,20,8.1
6,content_374e795aab68,0.863392,0,"false positive (flagged, but not actually decl...",High confidence -- model probability well abov...,235,20,31.0
7,content_b5e9e6453511,0.860493,1,correct (page is actually declining),High confidence -- model probability well abov...,157,20,6.1
8,content_87c007fb5c26,0.860091,1,correct (page is actually declining),High confidence -- model probability well abov...,2463,20,6.6
9,content_2bc3b7c8b3d9,0.857473,1,correct (page is actually declining),High confidence -- model probability well abov...,526,8,4.9


In [11]:
precision_at_20 = top20_model["true_label"].mean()
print(f"Top-20 precision: {precision_at_20:.3f} ({int(top20_model['true_label'].sum())} of 20 are actually declining)")
print(f"\\nOutcome breakdown:")
print(top20_model["outcome"].value_counts())

Top-20 precision: 0.750 (15 of 20 are actually declining)
\nOutcome breakdown:
outcome
correct (page is actually declining)                    15
false positive (flagged, but not actually declining)     5
Name: count, dtype: int64


**Reading the Top-20:** this is the queue an SEO/content owner would actually work down first. Precision here is the number that matters more than overall accuracy, because it directly answers "if I trust the top of this list, how often am I right?" Any false positives in this list are worth a quick manual sanity check before action -- consistent with the baseline notebook's own framing: a ranked list is a review queue, not proof.

## Self-check

- [x] Every section above is filled — markdown reasoning AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere (only anonymized IDs from the public starter dataset)
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Compared against the baseline on the same split; used a client-grouped holdout; explained method choice
- [x] Interprets features/errors; does not reward complexity alone (chose Logistic Regression deliberately, checked its errors rather than reaching for a bigger model first)
- [ ] Committed to my repo under `work/notebooks/` — then submit repo URL on the card. *(Do this last — see note below.)*